In [124]:
import numpy as np
import random

import pickle as pkl
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainwidemap import bwm_query, bwm_units, load_good_units, load_trials_and_mask
from tqdm import tqdm
from one.api import ONE
from brainbox.singlecell import bin_spikes2D
from iblatlas.regions import BrainRegions
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [125]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from ibl_info.utils import check_config
from ibl_info.rsa_regression import ideal_rsa_matrices, run_rsa_regression, simpler_rsa_matrices
from sklearn.linear_model import LinearRegression
from scipy.spatial.distance import pdist, squareform
from scipy.stats import zscore
from ibl_info.manifold import (
    plot_pcas_separate_decomposition,
    plot_pcas_separate_decomposition_adapted,
)
import warnings

warnings.filterwarnings("ignore")

### Only correct trajectories

In [ ]:
with open("../data/generated/manifold/bwm_accumulated_data.pkl", "rb") as f:
    dx = pkl.load(f)

In [ ]:
# normalization = False
# model_vectors, model_names = ideal_rsa_matrices()

In [ ]:
plot_pcas_separate_decomposition_adapted(
    dx, "LGd", ["Left Con", "Left Incon", "Right Con", "Right Incon"]
)

In [ ]:
# now we fit kernels

In [ ]:
model_vectors, model_names, model_matrices = simpler_rsa_matrices()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), ncols=3, sharex=True, sharey=True)

for idx, (name, array) in enumerate(model_matrices.items()):
    cond_labels = ["Left_Con", "Left_Incon", "Right_Con", "Right_Incon"]
    if idx == 2:
        cbar = True
    else:
        cbar = False
    sns.heatmap(
        array,
        ax=ax[idx],
        linewidths=0.3,
        square=True,
        xticklabels=cond_labels,
        yticklabels=cond_labels,
        cbar=False,
    )
    ax[idx].set_title(f"{name}")

In [ ]:
results_correct_only = run_rsa_regression(
    dx, model_vectors=model_vectors, model_names=model_names, conditions=4, model_type="NNLS"
)

In [ ]:
from ibl_info.rsa_regression import plot_rsa_dynamics

In [ ]:
plot_rsa_dynamics(results_correct_only, "GRN", model_names)

In [ ]:
for k in results_correct_only.keys():
    plot_rsa_dynamics(results_correct_only, k, model_names)

In [ ]:
from ibl_info.rsa_regression import plot_rsa_summary_bars

In [ ]:
df = plot_rsa_summary_bars(results_correct_only, model_names)

# Manifold, all trials


In [ ]:
with open("../data/generated/manifold/bwm_manifold_8conditions.pkl", "rb") as f:
    dx_all = pkl.load(f)

In [ ]:
model_vectors, model_names, model_matrices = ideal_rsa_matrices()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), ncols=5, sharex=True, sharey=True)
cond_labels = [
    "L_Cong_Corr",
    "L_Cong_Err ",
    "L_Inc_Corr ",
    "L_Inc_Err  ",
    "R_Cong_Corr",
    "R_Cong_Err ",
    "R_Inc_Corr ",
    "R_Inc_Err  ",
]
for idx, (name, array) in enumerate(model_matrices.items()):

    if idx == 2:
        cbar = True
    else:
        cbar = False
    sns.heatmap(
        array,
        ax=ax[idx],
        linewidths=0.3,
        square=True,
        xticklabels=cond_labels,
        yticklabels=cond_labels,
        cbar=False,
    )
    ax[idx].set_title(f"{name}")

In [ ]:
model_names

In [ ]:
results_all_without_outcome = run_rsa_regression(
    dx_all, model_vectors, ["Choice", "Prior", "Congruence", "Stimulus"], model_type="NNLS"
)

In [ ]:
results_all = run_rsa_regression(dx_all, model_vectors, model_names, model_type="NNLS")

In [ ]:
for k in results_correct_only.keys():
    plot_rsa_dynamics(results_all, k, model_names)

In [ ]:
plot_rsa_summary_bars(results_all, model_names)

In [ ]:
plot_rsa_summary_bars(results_all_without_outcome, ["Choice", "Prior", "Congruence", "Stimulus"])

In [99]:
from ibl_info.outcome_decoder import run_feedback_decoder
from iblatlas.atlas import BrainRegions
from one.api import ONE
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainwidemap.bwm_loading import merge_probes
from ibl_info.prepare_data_pid import (
    get_new_cinc_intervals,
    prepare_ephys_data,
    get_new_cinc_intervals_choice,
)

In [97]:
# plot probe output

subject_id = "CSH_ZAD_022"
eid = "a82800ce-f4e3-4464-9b80-4c3d6fade333"
session_id = eid
one = ONE()
ss = SessionLoader(one, eid=session_id)
ss.load_trials()

(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/zadorlab/Subjects/CSH_ZAD_022/2020-05-24/001/alf/_ibl_trials.intervals_bpod.npy: 100%|██████████| 10.9k/10.9k [00:00<00:00, 44.1kB/s]
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/zadorlab/Subjects/CSH_ZAD_022/2020-05-24/001/alf/_ibl_trials.quiescencePeriod.npy: 100%|██████████| 5.50k/5.50k [00:00<00:00, 18.8kB/s]
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/zadorlab/Subjects/CSH_ZAD_022/2020-05-24/001/alf/#2025-03-03#/_ibl_trials.stimOffTrigger_times.npy: 100%|██████████| 5.50k/5.50k [00:00<00:00, 20.0kB/s]
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/zadorlab/Subjects/CSH_ZAD_022/2020-05-24/001/alf/#2025-03-03#/_ibl_trials.stimOff_times.npy: 100%|██████████| 5.50k/5.50k [00:00<00:00, 19.3kB/s]
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/zadorlab/Subjects/CSH_ZAD_022/2020-05-24/001/alf/#2025-03-03#/_ibl_trials.stimOnTrigger_tim

In [114]:
pids, probes = one.eid2pid(session_id)
if isinstance(probes, list) and len(probes) > 1:
    to_merge = [load_good_units(one, pid=pid, qc=1) for pid in pids]
    spikes, clusters = merge_probes(
        [spikes for spikes, _ in to_merge], [clusters for _, clusters in to_merge]
    )
else:
    spikes, clusters = load_good_units(one, pid=pids[0], qc=1)

trials, mask = load_trials_and_mask(one, session_id, exclude_nochoice=True, exclude_unbiased=True)
trials = trials[mask]

# trials-feedback is the target
target_variable = trials["feedbackType"].values

# "Quiescent": {
#     "align": "stimOn_times",
#     "offset": -0.1,  # Align to -0.1s before Stim
#     "t_pre": 0.5,
#     "t_post": 0.0,
# }

stimon_times = trials.stimOn_times.values
qu_time_on = stimon_times - 0.5
qu_time_off = stimon_times - 0.1
intervals = np.array([qu_time_on, qu_time_off]).T

In [102]:
binned_spikes, actual_regions, n_units, cluster_uuids_list = prepare_ephys_data(
    spikes, clusters, intervals, ["VISa"], minimum_units=5
)  # this returns all neurons from a single region that pass qc

spike_data = binned_spikes[0]

Region found VISa, 26


In [ ]:
from ibl_info.outcome_decoder import run_decoder_nested_cv



Starting Nested CV (Outer Splits: 5)...


In [116]:
target_variable[target_variable == -1] = 0

In [117]:
target_variable

array([0., 1., 1., 1., 1., 1., 1., 0., 1., 1., 0., 1., 1., 1., 0., 1., 1.,
       0., 0., 1., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0.,
       0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 1., 0., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1.,
       1., 0., 1., 1., 1., 1., 0., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 0., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 0., 0., 1., 1., 1.,
       1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 1., 1.,
       1., 1., 1., 1., 1.

In [131]:
results = run_decoder_nested_cv(
    spike_data,
    target_variable,
    n_splits=5,
    scale=False,
)

Starting Nested CV (Outer Splits: 5)...


In [133]:
null_scores = []
y_shuffled = np.array(target_variable).copy().flatten()
n_permutations = 10
print(f"Starting {n_permutations} permutations...")

for i in range(n_permutations):
    print(f"  Permutation {i + 1}/{n_permutations}...")

    # SHUFFLE: Destroy the relationship between X and y
    np.random.shuffle(y_shuffled)

    # Run decoder on shuffled labels
    # Note: We pass neural_data unchanged, but y is shuffled
    perm_results = run_decoder_nested_cv(spike_data, y_shuffled)

    null_scores.append(perm_results["balanced_accuracy"])

null_scores = np.array(null_scores)

Starting 10 permutations...
  Permutation 1/10...
Starting Nested CV (Outer Splits: 5)...
  Permutation 2/10...
Starting Nested CV (Outer Splits: 5)...
  Permutation 3/10...
Starting Nested CV (Outer Splits: 5)...
  Permutation 4/10...
Starting Nested CV (Outer Splits: 5)...
  Permutation 5/10...
Starting Nested CV (Outer Splits: 5)...
  Permutation 6/10...
Starting Nested CV (Outer Splits: 5)...
  Permutation 7/10...
Starting Nested CV (Outer Splits: 5)...
  Permutation 8/10...
Starting Nested CV (Outer Splits: 5)...
  Permutation 9/10...
Starting Nested CV (Outer Splits: 5)...
  Permutation 10/10...
Starting Nested CV (Outer Splits: 5)...


In [134]:
results["balanced_accuracy"]

0.49922485071198897

In [135]:
np.mean(null_scores)

np.float64(0.5082165824529169)